<!-- SPDX-FileCopyrightText: 2026 Orbital Research Cluster for Celestial Applications (ORCCA) Lab, University of Colorado at Boulder -->
<!-- SPDX-License-Identifier: ISC -->
# OSIRIS-REx Tutorial Series
---
*Scarabaeus | Last revised 2026*

## Objectives
This tutorial walks through a complete orbit determination (OD) scenario within Scarabaeus (SCB) using real data from the OSIRIS-REx mission

## 1.0 - Load SCB and the relevant Python packages

Before doing anything else, import the libraries the tutorial depends on:

- **`scarabaeus`** — the main OD library. Aliased to `scb` throughout for brevity.
- **`numpy`** — array mathematics used heavily inside SCB and in the pre-processing steps below.
- **`matplotlib`** — plotting library used to visualize residuals and filter results in later cells.
- **`sklearn.linear_model.HuberRegressor`** — a robust linear regression estimator used in Section 1.4 to detrend measurement residuals before outlier rejection.
- **`PCInfo` / `constants`** — SCB sub-modules that expose planetary constants and SPICE body identifiers.

In [1]:
import os 
import scarabaeus as scb 
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker  
from sklearn import linear_model
import supplementary as supp

## 1.1 - Initialize the units, frames, and relevant kernels

### Units
SCB tracks physical units explicitly through the `ArrayWUnits` class, which wraps a numeric array together with its dimensional unit. Units are retrieved by name via `scb.Units.get_units()`. Arithmetic on `ArrayWUnits` objects propagates units automatically, catching dimension mismatches at runtime rather than producing silently wrong numbers. The four units retrieved here — **kg**, **km**, **sec**, and **unitless** — are the building blocks for every physical quantity in this tutorial.

### Reference Frames
A `Frame` object in SCB identifies a coordinate reference frame and is attached to every state vector, position, and velocity throughout the OD pipeline so that automatic conversions can be applied when needed. `scb.Frame.generate_common_frames()` returns the four frames used most often:

| Frame | Description |
|---|---|
| `J2000` | Earth Mean Equator and Equinox of J2000.0 (inertial) — the default propagation frame |
| `ITRF93` | International Terrestrial Reference Frame — body-fixed frame for Earth ground stations |
| `ECLIPJ2000` | Mean Ecliptic and Equinox of J2000.0 (inertial) |
| `IAUEARTH` | IAU Earth-fixed frame |

### SPICE Kernels
SCB uses NAIF SPICE under the hood to evaluate solar system geometry (positions, orientations, and time conversions). All kernel files are loaded at once through a *metakernel* — a plain-text file that lists the paths of every SPK, PCK, LSK, and frame kernel required for this scenario. `scb.SpiceManager.print_kernels()` confirms what is currently loaded.

Key parameters for the OSIRIS-REx scenario:
- **`sc_id = -64`** — the NAIF integer ID for the OSIRIS-REx spacecraft.
- **`scft_transpd_delay`** — the spacecraft transponder delay in Range Units (RU), a hardware calibration constant that must be subtracted when converting raw sequential ranging measurements to light-travel-time distances.

### Spacecraft and CelestialBody
`scb.Spacecraft` bundles the satellite's name, SPICE ID, mass, cross-sectional area, and solar radiation pressure (SRP) reflectivity coefficient into a single object that is passed to the force model and measurement models later. `scb.CelestialBody.from_constants("SUN")` creates the gravitational center — the Sun — for this heliocentric scenario.

In [2]:
# load tutorial data
data = supp.load_data()

# load kernels
scb.SpiceManager.clear_kernels()
scb.SpiceManager.load_kernel_from_mkfile(data.OREx_real_data_mk.path)


# Generate common frames and units
kg, km, sec, unitless = scb.Units.get_units(["kg", "km", "sec", "unitless"])
(J2000, ITRF93, ECLIPJ2000, IAUEARTH) = scb.Frame.generate_common_frames()

# Initialize the kernels
sc_id = -64
scft_transpd_delay = 16529.719234434557

# Initialize the body object for the orbiter
Orbiter_mass = scb.ArrayWUnits(1200.0, kg)
Orbiter_area = scb.ArrayWUnits(1.2e-05, km**2)
Orbiter_cr_srp = scb.ArrayWUnits(0.8, None)
#
Orbiter = scb.Spacecraft("Orbiter", sc_id, Orbiter_mass, Orbiter_area, Orbiter_cr_srp)

# Initialize frame, planet constants and origin
frame = J2000
origin = scb.CelestialBody.from_constants("SUN")

SCB supplementary data up to date.


## 1.2 - Load the radiometric tracking data for OREx and apply media corrections

### Radiometric Tracking Data
The Deep Space Network (DSN) generates two types of radiometric observables used for orbit determination:

- **Sequential Ranging (SR)** — a pseudo-noise ranging tone sequence is uplinked from the ground station, turned around by the spacecraft transponder, and received back at the station. The round-trip light time is extracted and converted to a range measurement.
- **Doppler** — the frequency shift between the transmitted and received carrier signal is measured over a count interval `Tc`. This yields an average range-rate (radial velocity) observable.

Both observables are stored in raw SFDU (Standard Formatted Data Unit) records. The `aux_fun()` helper defined here extracts all the ancillary metadata — uplink/downlink frequencies, station calibration delays, ramp table entries, modulo settings, and outlier flags — from the SFDU records and packages them into dictionaries consumed by the SCB measurement models.

### Ground Station
The DSN station that tracked OSIRIS-REx during this pass is identified automatically from the tracking file. `scb.GroundStation` looks up the station's ECEF coordinates and SPICE ID from the loaded kernels so that the one-way light-time from spacecraft to station can be computed at each observation epoch.

### Measurement Models
`scb.SequentialRangingReal` and `scb.DopplerReal` are the measurement model classes for real (non-simulated) data. Each model holds a reference to the ground station, the a-priori measurement noise standard deviation (`sigma`), an optional measurement bias, and the auxiliary data dictionary. The `observed_measurements()` method reads the raw observable values from the tracking file and wraps them in SCB's `ArrayWFrame` type.

### Media Corrections
Radio signals passing through Earth's atmosphere accumulate path-length errors from:

- **Troposphere** — a non-dispersive delay that affects all frequencies. SCB applies both a seasonal component (from a pre-computed file) and a non-seasonal component (found by `find_tro_metadata()` for the specific pass date).
- **Ionosphere** — a dispersive, frequency-dependent delay. SCB finds the appropriate ionosphere correction file via `find_ion_metadata()`.

`scb.MediaCorrections` objects are attached directly to the measurement models, so corrections are applied automatically when computed measurements are generated during filtering.

In [3]:
def aux_fun(trk_data):
    # Generate the sranging auxiliary data dictionary
    sranging_aux_data = {
        #
        # Sranging auxiliary data (from 2W_sranging, SFDU_7)
        #
        "RU": (((np.array(trk_data["aux_SFDU_7"]["exc_scalar_den"]) / np.array(trk_data["aux_SFDU_7"]["exc_scalar_num"])) / 16 ) ** -1).tolist(),
        "M": trk_data["aux_SFDU_7"]["rng_modulo"],
        "M2_num": trk_data["aux_SFDU_16"]["M2_num"],
        "M2_den": trk_data["aux_SFDU_16"]["M2_den"],
        "rng_type": trk_data["aux_SFDU_7"]["rng_type"],
        "outlier_flag_sranging": trk_data["aux_SFDU_7"]["outlier_flag"],
        "year_SFDU7": trk_data["aux_SFDU_7"]["year"],
        "doy_SFDU7": trk_data["aux_SFDU_7"]["doy"],
        "sec_SFDU7": trk_data["aux_SFDU_7"]["sec"],
        #
        # Delays auxiliary data (from 2W_sranging, SFDU_7)
        #
        "ul_freq": trk_data["aux_SFDU_7"]["ul_freq"],
        "ul_zheight_corr": trk_data["aux_SFDU_2"]["ul_zheight_corr"],
        "dl_zheight_corr": trk_data["aux_SFDU_3"]["dl_zheight_corr"],
        "scft_transpd_delay": scft_transpd_delay, # [RU]  
        "scft_transpd_delay_s": scft_transpd_delay / ((221.0 / (749.0 * 2.0)) * 8445.767679 * 1e6), # [s] 
        "transmit_time_tag_delay": trk_data["2W_doppler"]["transmit_time_tag_delay"],
        "rcv_time_tag_delay": trk_data["2W_doppler"]["rcv_time_tag_delay"],
        "array_delay": trk_data["2W_doppler"]["array_delay"],
        "ul_stn_cal": trk_data["aux_SFDU_7"]["ul_stn_cal"],
        "dl_stn_cal": trk_data["aux_SFDU_7"]["dl_stn_cal"],
        "exc_scalar_num": trk_data["aux_SFDU_7"]["exc_scalar_num"],
        "exc_scalar_den": trk_data["aux_SFDU_7"]["exc_scalar_den"],
        #
        # Ramp table auxiliary data (from SFDU_9)
        #
        "ramp_sec": trk_data["aux_SFDU_9"]["sec"],
        "ramp_day": trk_data["aux_SFDU_9"]["doy"],
        "ramp_freq": trk_data["aux_SFDU_9"]["ramp_freq"],
        "ramp_rate": trk_data["aux_SFDU_9"]["ramp_rate"],
        "ramp_type": trk_data["aux_SFDU_9"]["ramp_type"],
    }
    Doppler_aux_data = {
        #
        # Doppler auxiliary data (from 2W_doppler, SFDU_16)
        #
        "Tc": trk_data["aux_SFDU_16"]["Tc"],
        "M2_num": trk_data["aux_SFDU_16"]["M2_num"],
        "M2_den": trk_data["aux_SFDU_16"]["M2_den"],
        "outlier_flag_doppler": trk_data["aux_SFDU_16"]["outlier_flag"],
        "year_SFDU16": trk_data["aux_SFDU_16"]["year"],
        "doy_SFDU16": trk_data["aux_SFDU_16"]["doy"],
        "sec_SFDU16": trk_data["aux_SFDU_16"]["sec"],
        #
        # Delays auxiliary data (from 2W_doppler, SFDU_16)
        #
        "ul_zheight_corr": trk_data["aux_SFDU_2"]["ul_zheight_corr"],
        "dl_zheight_corr": trk_data["aux_SFDU_3"]["dl_zheight_corr"],
        "scft_transpd_delay": scft_transpd_delay, # [RU]  
        "scft_transpd_delay_s": scft_transpd_delay / ((221.0 / (749.0 * 2.0)) * 8445.767679 * 1e6), # [s] 
        "transmit_time_tag_delay": trk_data["2W_doppler"]["transmit_time_tag_delay"],
        "rcv_time_tag_delay": trk_data["2W_doppler"]["rcv_time_tag_delay"],
        "array_delay": trk_data["2W_doppler"]["array_delay"],
        #
        # Method (1) - Take these values from 2W_sranging and aux_SFDU_7, just max-unique, since its asynch with Doppler time
        #
        "ul_freq": (np.array(trk_data["aux_SFDU_2"]["ul_zheight_corr"]) * 0).tolist() + np.max(np.unique(trk_data["aux_SFDU_7"]["ul_freq"])),       
        "ul_stn_cal": (np.array(trk_data["aux_SFDU_2"]["ul_zheight_corr"]) * 0).tolist() + np.max(np.unique(trk_data["aux_SFDU_7"]["ul_stn_cal"])),
        "dl_stn_cal": (np.array(trk_data["aux_SFDU_2"]["ul_zheight_corr"]) * 0).tolist() + np.max(np.unique(trk_data["aux_SFDU_7"]["dl_stn_cal"])),
        "exc_scalar_num": (np.array(trk_data["aux_SFDU_2"]["ul_zheight_corr"]) * 0).tolist() + np.max(np.unique(trk_data["aux_SFDU_7"]["exc_scalar_num"])),
        "exc_scalar_den": (np.array(trk_data["aux_SFDU_2"]["ul_zheight_corr"]) * 0).tolist() + np.max(np.unique(trk_data["aux_SFDU_7"]["exc_scalar_den"])),
        "RU": 0.14753004005340453, #
        #
        # Ramp table auxiliary data (from SFDU_9)
        #
        "ramp_sec": trk_data["aux_SFDU_9"]["sec"],
        "ramp_day": trk_data["aux_SFDU_9"]["doy"],
        "ramp_freq": trk_data["aux_SFDU_9"]["ramp_freq"],
        "ramp_rate": trk_data["aux_SFDU_9"]["ramp_rate"],
        "ramp_type": trk_data["aux_SFDU_9"]["ramp_type"],
    }
    #
    return sranging_aux_data, Doppler_aux_data

def add_media_to_meas_model(obs_quantities, meas_model, media_correction_object):
    #
    year_pass_media, day_pass_media, _ = scb.SpiceManager.et2YDS([obs_quantities[0].times.values[0]])
    metadata_tropo = scb.MediaCorrections.find_tro_metadata(tropo_file_bucket, year_pass_media[0], day_pass_media[0])
    metadata_iono = scb.MediaCorrections.find_ion_metadata(iono_file_bucket, year_pass_media[0], day_pass_media[0])
    #
    if metadata_tropo:
        media_correction_object.tropo_non_seasonal_file_path = os.path.join(tropo_file_bucket,metadata_tropo['file'])
        #
    if metadata_iono:
        media_correction_object.iono_file_path = os.path.join(iono_file_bucket,metadata_iono['file'])
    #
    meas_model.media_corrections = media_correction_object
    #
    return meas_model
#

# Setup tropo correction filepaths
tropo_seasonal_file_path = data.tropo_seasonal.path
trk_file_path = data.trk_file.path
#
tropo_file_bucket = "data/measurements/media_correction"
iono_file_bucket = "data/measurements/media_correction"

# Load the radiometric data 
meas_dict_list = []
#

trk_data = scb.Utils.load_json(trk_file_path)
aux_data_sr, aux_data_dop = aux_fun(trk_data)
#
meas_dict_list.append(aux_data_sr)
meas_dict_list.append(aux_data_dop)

# Get the ground station
GS1_name = "DSS-" + str(trk_data["2W_doppler"]["spice_id"][0])  # Extract GS_name
GS1 = scb.GroundStation(GS1_name)

sranging_sigma = scb.ArrayWUnits(1e1, km / km)
doppler_sigma = scb.ArrayWUnits(1e-3, 1 / sec)

# Create the measurement real measurement models for the selected pass 
sranging_model = scb.SequentialRangingReal("GS1 Real Range", GS1, sigma = sranging_sigma,
                                                 meas_bias = 0.0, computed_measurements_dict = meas_dict_list[0])
#
doppler_model  = scb.DopplerReal("GS1 Real Range Rate", GS1, sigma = doppler_sigma,  
                                     meas_bias = 0.0, computed_measurements_dict = meas_dict_list[1])

# Create the media correction objects 
mc_sranging = scb.MediaCorrections(name = "GS1 Media Corrections srang", instrument = GS1,
                                   tropo_seasonal_file_path = tropo_seasonal_file_path)
#
mc_doppler  = scb.MediaCorrections(name = "GS1 Media Corrections doppler", instrument = GS1,
                                   tropo_seasonal_file_path = tropo_seasonal_file_path)
#
# Generate observed quantities
sr_filename = trk_file_path
dop_filename = trk_file_path
#
sr_obs = sranging_model.observed_measurements(sr_filename, meas_name="2W_sranging", units= km / km)
doppler_obs = doppler_model.observed_measurements(dop_filename, meas_name="2W_doppler", units= 1 / sec)

#
# Add media correction to the measurement models 
sranging_model = add_media_to_meas_model(sr_obs, sranging_model, mc_sranging)
doppler_model = add_media_to_meas_model(doppler_obs, doppler_model, mc_doppler)

#
# Organize products for processing 
obs_quantities_list = [sr_obs, doppler_obs]
model_list = [sranging_model, doppler_model]
meas_type_list = ['range', 'range_rate']
file_label_list = ["real_range", "real_range_rate"]


FileNotFoundError: [Errno 2] No such file or directory: 'data/measurements/media_correction'

## 1.3 - Define the orbital scenario and propagate the reference trajectory

### Time Window
The scenario time window is derived directly from the observation epochs so that the propagated trajectory spans the full tracking pass. A one-hour pad is added to each end (`et_min_pad`, `et_max_pad`) so the integrator has settled before the first measurement and runs past the last one. The epoch array is then sampled at one million evenly spaced points — a fine enough resolution for accurate interpolation of the trajectory during filter processing.

### State Initialization
The initial state (position and velocity in `J2000` with the Sun as origin) is pulled from SPICE using `scb.SpiceManager.get_state()`. This uses the reconstructed OSIRIS-REx SPK kernels loaded in Section 1.1, giving a high-quality starting point at `et_start`. The `scb.StateArray` and `scb.StateDefinition` classes bundle the initial conditions along with metadata — which parameters are *estimated* versus *fixed*, and which body they belong to — into a structured state vector that the filter will later augment with solve-for parameters.

### Force Model
`scb.ForceModelTranslation` constructs the equations of motion used by the integrator. For this scenario, the force model includes:

- **Point-mass gravity** from the Sun (the primary body)
- **Third-body perturbations** from all eight planets plus Pluto, referenced by their NAIF barycenter IDs
- **Cannonball solar radiation pressure (SRP)** — a simplified model that treats the spacecraft as a sphere with a fixed reflectivity coefficient and cross-sectional area

### Propagation and SPK Output
`scb.Propagator` drives a DOP853 (Dormand-Prince 8th-order) Runge-Kutta integrator over the full epoch array. After propagation, the trajectory is wrapped in a `scb.Trajectory` object and written to a new SPICE SPK kernel (`orbiter_true.bsp`). This kernel acts as the *truth* reference trajectory — subsequent tutorial parts will compare filter estimates against it to assess OD accuracy.

In [4]:
#
# Get the time period for the scenario 
et_min = 1e34
et_max = 0.0
for obs in obs_quantities_list:
    et = obs[0]
    et_min = np.min([np.min(et.times.values), et_min])
    et_max = np.max([np.max(et.times.values), et_max])

#
# Collect all ET values into one array
all_et_obs = np.concatenate([obs[0].times.values for obs in obs_quantities_list])
all_et_obs_unique = np.sort(np.unique(all_et_obs))

et_min_obs, et_max_obs = all_et_obs_unique[0], all_et_obs_unique[-1]
et_min_pad = et_min_obs - 3600
et_max_pad = et_max_obs + 3600
et_padded = np.linspace(et_min_pad, et_max_pad, int(1e6)) 

et_start = et_padded[0]
et_end = et_padded[-1]
et_prop = et_padded

print('Start time (UTC): ', scb.SpiceManager.et2utc(et_start), ' Start time (ET): ', et_start)
print('End time (UTC): ', scb.SpiceManager.et2utc(et_end), ' End time (ET): ', et_end)

epoch_array = scb.EpochArray(et_prop, sys = "TDB")
epoch_array_obs = scb.EpochArray(all_et_obs_unique, sys = "TDB") 

#
# Propagate the nominal true trajectory 
sc_state_0 = scb.SpiceManager.get_state(str(sc_id), et_start, J2000, origin).values
pos_0 = scb.ArrayWFrame(sc_state_0[0:3], km, J2000)
vel_0 = scb.ArrayWFrame(sc_state_0[3:6], km/sec, J2000)

state = [
     ("position", 3, "estimated", "dynamic", Orbiter, pos_0),
     ("velocity", 3, "estimated", "dynamic", Orbiter, vel_0),
]

state_vector = scb.StateArray(epoch=epoch_array[0], origin=origin, state=scb.StateDefinition.from_components(state))
orbiter_true_filepath = os.getcwd() + "/data/kernels/scenario/orbiter_true.bsp"
if os.path.isfile(orbiter_true_filepath):
     os.remove(orbiter_true_filepath)

#
# Propagate true trajectory
third_bodies = [
     "MERCURYBARYCENTER",
     "VENUSBARYCENTER",
     "EARTHBARYCENTER",
     "MARSBARYCENTER",
     "JUPITERBARYCENTER",
     "SATURNBARYCENTER",
     "URANUSBARYCENTER",
     "NEPTUNEBARYCENTER",
     "PLUTOBARYCENTER",
]

# Initialize the force model and propagate
force_model = scb.ForceModelTranslation(primary_body=Orbiter, 
                                         third_bodies = third_bodies, 
                                         cannonball_SRP = True)

prop = scb.Propagator(
     primary_body=Orbiter,
     state_vector=state_vector,
     tspan=epoch_array,
     force_models = force_model
)

prop.propagate()
state_propagated = prop.propagated_state_array

# Save true trajectory
orbiter_traj = scb.Trajectory(state_array=prop.propagated_state_array)
orbiter_traj.write_to_spk('data/kernels/scenario/orbiter_true.bsp')

NameError: name 'obs_quantities_list' is not defined

## 1.4 - Pre-process measurements with coarse thresholding

Before the measurements are passed to the filter, gross outliers must be removed. A single bad data point can corrupt the filter's covariance or even cause divergence, so it is standard practice in OD to apply a coarse editing pass prior to sequential processing.

### Why Huber Regression?
A simple approach would be to compute residuals (observed minus computed), fit a polynomial trend, and flag points far from the trend. However, ordinary least squares trend fits are themselves pulled toward outliers, defeating the purpose. Instead, the `clean_obs_sr()` and `clean_obs_dop()` functions use **Huber regression** — a robust estimator that down-weights large residuals during the fit. This produces a detrended residual sequence that is insensitive to the very outliers we want to flag.

### Median Absolute Deviation (MAD) Thresholding
Once the Huber-detrended residuals are computed, outliers are identified using the **Median Absolute Deviation (MAD)**, a robust measure of spread:

```
MAD = median(|r_i - median(r)|)
```

Any observation whose detrended residual exceeds `mad_threshold × MAD` is discarded. The thresholds used here (`thresh_sr = 5.0`, `thresh_dop = 30.0`) are intentionally loose — this is a *coarse* editing step meant to remove only the most egregious outliers, not to perform the fine-grained data editing that the filter itself will apply pass-by-pass.

### After Cleaning
The cleaned observation tuples and their corresponding auxiliary data dictionaries are used to rebuild the measurement models with tighter noise sigmas (`σ_range = 0.5 RU`, `σ_doppler = 0.001 Hz`). The updated model list, observation list, and type labels are what will be handed to the filter in Part 5.

In [5]:
def clean_obs_sr(
    obs,
    model,
    target,
    frame,
    aux_data,
    mad_threshold
):
    """
    Remove outliers using Huber-detrended residuals.

    Parameters
    ----------
    obs : tuple
        Observation tuple: (epoch_array, ..., observed_values, ...)
    model : object
        Measurement model.
    target : object
        Target object.
    frame : object
        Measurement frame.
    aux_data : dict
        Auxiliary measurement data passed into eps_override.
    mad_threshold : float
        Threshold multiplier for MAD-based outlier rejection.

    Returns
    -------
    obs_clean : tuple
        Cleaned observations.
    aux_data_clean : dict
        Cleaned auxiliary data corresponding to retained observations.
    """

    _, _, _, computed_obs = model.computed_measurements(
        target=target,
        epoch_array=obs[0],
        frame=frame,
        noisy=False,
        eps_override=aux_data,
    )

    obs_vals = obs[2].quantity.values
    computed_vals = computed_obs.quantity.values
    resids = obs_vals - computed_vals

    x_huber = obs[0].times.values
    y_huber = resids

    clf = linear_model.HuberRegressor()
    clf.fit(x_huber[:, np.newaxis], y_huber)
    clf_evals = clf.predict(x_huber[:, np.newaxis])

    resids_huber = resids - clf_evals

    med_resids_huber = np.median(resids_huber)
    abs_devs_huber = np.abs(resids_huber - med_resids_huber)
    mad = np.median(abs_devs_huber)

    mask = np.abs(resids_huber) <= mad_threshold * mad

    time_vals_clean = obs[0].times.values[mask]
    obs_values_clean = obs[2].quantity.values[mask]

    obs_times_clean = scb.EpochArray(
        time_vals_clean,
        sys="TDB",
    )

    obs_vals_clean = scb.ArrayWFrame(
        obs_values_clean,
        km / km,
        frame,
    )

    obs_clean = (
        obs_times_clean,
        obs[1][mask],
        obs_vals_clean,
        obs[3][mask],
    )

    aux_data_clean = aux_data.copy()

    for kk in aux_data_clean:
        if isinstance(aux_data_clean[kk], list):
            if len(aux_data_clean[kk]) == len(mask):
                temp = np.array(aux_data_clean[kk])[mask]
                aux_data_clean[kk] = temp.tolist()

    return obs_clean, aux_data_clean

def clean_obs_dop(
    obs,
    model,
    target,
    frame,
    aux_data,
    mad_threshold
):
    """
    Remove outliers using Huber-detrended residuals.

    Parameters
    ----------
    obs : tuple
        Observation tuple: (epoch_array, ..., observed_values, ...)
    model : object
        Measurement model.
    target : object
        Target object.
    frame : object
        Measurement frame.
    aux_data : dict
        Auxiliary measurement data passed into eps_override.
    mad_threshold : float
        Threshold multiplier for MAD-based outlier rejection.

    Returns
    -------
    obs_clean : tuple
        Cleaned observations.
    aux_data_clean : dict
        Cleaned auxiliary data corresponding to retained observations.
    """

    _, _, _, computed_obs = model.computed_measurements(
        target=target,
        epoch_array=obs[0],
        frame=frame,
        noisy=False,
        eps_override=aux_data,
    )

    obs_vals = obs[2].quantity.values
    computed_vals = computed_obs.quantity.values
    resids = obs_vals - computed_vals

    x_huber = obs[0].times.values
    y_huber = resids

    clf = linear_model.HuberRegressor()
    clf.fit(x_huber[:, np.newaxis], y_huber)
    clf_evals = clf.predict(x_huber[:, np.newaxis])

    resids_huber = resids - clf_evals

    med_resids_huber = np.median(resids_huber)
    abs_devs_huber = np.abs(resids_huber - med_resids_huber)
    mad = np.median(abs_devs_huber)

    mask = np.abs(resids_huber) <= mad_threshold * mad

    time_vals_clean = obs[0].times.values[mask]
    obs_values_clean = obs[2].quantity.values[mask]

    obs_times_clean = scb.EpochArray(
        time_vals_clean,
        sys="TDB",
    )

    obs_vals_clean = scb.ArrayWFrame(
        obs_values_clean,
        1 / sec,
        frame,
    )

    obs_clean = (
        obs_times_clean,
        obs[1][mask],
        obs_vals_clean,
        obs[3][mask],
    )

    aux_data_clean = aux_data.copy()

    for kk in aux_data_clean:
        if isinstance(aux_data_clean[kk], list):
            if len(aux_data_clean[kk]) == len(mask):
                temp = np.array(aux_data_clean[kk])[mask]
                aux_data_clean[kk] = temp.tolist()

    return obs_clean, aux_data_clean


thresh_sr = 5.0
thresh_dop = 30.0
#
sr_obs_1, aux_data_sr_1 = clean_obs_sr(sr_obs, sranging_model, Orbiter, frame, meas_dict_list[0], thresh_sr)
doppler_obs_1, aux_data_doppler_1 = clean_obs_dop(doppler_obs, doppler_model, Orbiter, frame, meas_dict_list[1], thresh_dop)

meas_dict_list[0] = aux_data_sr_1
meas_dict_list[1] = aux_data_doppler_1

sranging_sigma = scb.ArrayWUnits(0.5, km/km)
doppler_sigma  = scb.ArrayWUnits(0.001, 1 / sec)

#
# Create the measurement real measurement models for the selected pass 
sranging_model_1 = scb.SequentialRangingReal("GS1 Real Range", GS1, sigma=sranging_sigma,
                                                 meas_bias = 0.0, 
                                                 computed_measurements_dict = meas_dict_list[0])
#
doppler_model_1  = scb.DopplerReal("GS1 Real Range Rate", GS1,sigma=doppler_sigma,  
                                     meas_bias = 0.0, 
                                     computed_measurements_dict = meas_dict_list[1])
#
# Generate media correction objects
mc_sranging_1 = scb.MediaCorrections(name = "GS1 Media Corrections srang",instrument = GS1,
                                   tropo_seasonal_file_path= tropo_seasonal_file_path)

mc_doppler_1 = scb.MediaCorrections(name = "GS1 Media Corrections srang",instrument = GS1,
                                   tropo_seasonal_file_path= tropo_seasonal_file_path)

#
# Add media correction to meas models
sranging_model_1 = add_media_to_meas_model(sr_obs_1, sranging_model_1, mc_sranging_1)
doppler_model_1 = add_media_to_meas_model(doppler_obs_1, doppler_model_1, mc_doppler_1)

#
# Create the lists used for filtering 
obs_quantities_list = [sr_obs_1, doppler_obs_1]
model_list = [sranging_model_1, doppler_model_1]
meas_type_list = ["range", "rangerate"]
file_label_list = ["real_range", "real_range_rate"]


Generating computed measurements...


2W Sequential Ranging:   0%|                                                    | 0/99 obs [00:00<?]

2W Sequential Ranging:   1%|▍                                               | 1/99 obs [00:00<00:01]

2W Sequential Ranging:   2%|▉                                               | 2/99 obs [00:00<00:01]

2W Sequential Ranging:   3%|█▍                                              | 3/99 obs [00:00<00:01]

...

2W Doppler: 100%|███████████████████████████████████████████████████████▋| 216/217 obs [00:00<00:00]

2W Doppler: 100%|████████████████████████████████████████████████████████| 217/217 obs [00:00<00:00]

2W Doppler: 100%|████████████████████████████████████████████████████████| 217/217 obs [00:00<00:00]

2W Doppler: 100%|████████████████████████████████████████████████████████| 217/217 obs [00:00<00:00]

Exception: Argument [values] must be a non-empty numpy.ndarray instance.

## 1.5 - Filtering

With the reference trajectory propagated and the measurements cleaned, we are ready to run the orbit determination filter.

### Filter Type: SRIF
SCB implements the **Square Root Information Filter (SRIF)**, a numerically stable variant of the classical Kalman filter. Rather than propagating the covariance matrix `P` directly, the SRIF propagates its square-root information form, which avoids the positive-definiteness loss that can occur in standard implementations when process noise is small or the problem is nearly singular. This makes it well-suited for deep-space OD where measurement arcs are long and the dynamical system is stiff.

### Reference Trajectory and Initial Perturbation
The filter operates on deviations from a *reference trajectory*, not on the absolute state. A new `Spacecraft` object (`Orbiter_pert_it1`) is created for this reference, given a distinct SPICE ID (`-1001`) so its trajectory can be written to a separate SPK kernel without conflicting with the truth. The initial position and velocity perturbations (`ic_pos_delta`, `ic_vel_delta`) are set to zero here, meaning the filter starts from the SPICE truth — a useful baseline for validating that the filter converges correctly before introducing intentional errors in later experiments.

### Initial Covariance `P`
`scb.CovarianceMatrix` constructs the 6×6 diagonal initial state covariance from a list of per-component sigmas:

| Component | Sigma | Interpretation |
|---|---|---|
| Position (x, y, z) | 1000 km | Large uncertainty — the filter is given significant freedom in the position estimate |
| Velocity (vx, vy, vz) | 1×10⁻³ km/s | Tighter velocity uncertainty |

These values are deliberately conservative. The filter will reduce the covariance as it processes measurements.

### Process Noise `Q`
The process noise covariance `Q` accounts for unmodeled accelerations (thruster firings, outgassing, solar radiation pressure model errors, etc.) that cause the true trajectory to deviate from the deterministic force model over time. SCB uses **Stochastic Noise Compensation (SNC)**, a continuous-time model that adds a small random acceleration in each Cartesian direction. `Q_cont` is the power spectral density of that acceleration noise, specified in km²/s⁴.

### Filter Settings and Execution
`scb.FilterSettings` bundles the covariance and process noise configuration together with output metadata. The `scb.SRIF` object is then initialized with the reference propagator, the settings, and the cleaned measurement list assembled from Section 1.4.

Calling `srif_it1.fit()` runs the filter forward through the tracking pass, processing each sequential ranging and Doppler observation in time order. After each measurement update, the state estimate and covariance are refined. The `max_iterations` and `convergence_threshold` parameters control an outer iteration loop — the filter can be re-linearized around the updated trajectory and re-run until the state correction falls below the threshold, which improves accuracy when the initial reference is far from the truth.

In [6]:
#
# Initialize some parameters 
ic_pos_delta         = np.array([0.0, 0.0, 0.0]) # [km]
ic_vel_delta         = np.array([0.0, 0.0, 0.0]) # [km/sec]
filter_pos_sigma     = 1000 # [km]
filter_vel_sigma     = 1e-3 # [km/sec]
filter_process_noise = 1e-11 # [km^2 / sec^4] # THESE ARE NOT THE CORRECT UNITS!!
filter_iterations    = 1 
filter_convergence_thr = 1 # [-] # WHAT IS THIS?
filter_type          = 'SRIF'

#
# Generated the predicted orbit for the initial guess
Orbiter_pert_it1 = scb.Spacecraft("orbiter_ref_it1", -1001, Orbiter_mass, Orbiter_area, Orbiter_cr_srp)

antenna_sc = scb.Antenna("Antenna_for_radiometric", spice_id=-1000)
Orbiter_pert_it1.add_instrument([antenna_sc])

# Perturb true to get reference it-1
delta_pos = scb.ArrayWUnits(ic_pos_delta, km)
delta_vel = scb.ArrayWUnits(ic_vel_delta, km / sec)
pos_pert_it1 = scb.ArrayWFrame(pos_0.quantity + delta_pos, frame)
vel_pert_it1 = scb.ArrayWFrame(vel_0.quantity + delta_vel, frame)

state_pert_it1 = [
    ("position", 3, "estimated", "dynamic", Orbiter_pert_it1, pos_pert_it1),
    ("velocity", 3, "estimated", "dynamic", Orbiter_pert_it1, vel_pert_it1),
]

state_vector_pert = scb.StateArray(epoch=epoch_array[0], origin=origin, state=scb.StateDefinition.from_components(state_pert_it1))

# Check existence of trajectory kernels
orbiter_est_filepath = os.getcwd() + "/data/kernels/scenario/orbiter_ref_it1.bsp"
if os.path.isfile(orbiter_est_filepath):
    os.remove(orbiter_est_filepath)

# Propagate reference trajectory at it-1
third_bodies = [
    "MERCURYBARYCENTER",
    "VENUSBARYCENTER",
    "EARTHBARYCENTER",
    "MARSBARYCENTER",
    "JUPITERBARYCENTER",
    "SATURNBARYCENTER",
    "URANUSBARYCENTER",
    "NEPTUNEBARYCENTER",
    "PLUTOBARYCENTER"
]

force_model_pert_it1 = scb.ForceModelTranslation(primary_body=Orbiter_pert_it1, 
                                                    third_bodies = third_bodies, 
                                                    cannonball_SRP = True)
#
prop_pert_it1 = scb.Propagator(
    primary_body=Orbiter_pert_it1,
    state_vector=state_vector_pert,
    tspan=epoch_array,
    force_models = force_model_pert_it1
)

# Set up initial covariance matrix P (valid also for it-2, and it-3)
pos_sigma = scb.ArrayWUnits(filter_pos_sigma, km)
vel_sigma = scb.ArrayWUnits(filter_vel_sigma, km / sec)
state_sigma_list = [pos_sigma, pos_sigma, pos_sigma, vel_sigma, vel_sigma, vel_sigma]
#
state_covariance = scb.CovarianceMatrix(
    state_sigma_list, epoch_array[0], from_list=True
)

# Set up process noise covariance Q
Q_diag = scb.ArrayWUnits(filter_process_noise, km**2 * sec**-4) # Not right units
Q_sigma_list = [Q_diag, Q_diag, Q_diag]
Q_cont = scb.CovarianceMatrix(
    Q_sigma_list, epoch_array[0], from_list=True
)

# Make list of measurements for filtering 
measurements_list_new = []
#
measurements_list_new.append({
    "model":model_list[0], 
    "measurement_type":meas_type_list[0],
    "observed_meas":obs_quantities_list[0],
    "dataset_name":model_list[0].name,
    "file_label":file_label_list[0]
    })
#
measurements_list_new.append({
    "model":model_list[1], 
    "measurement_type":meas_type_list[1],
    "observed_meas":obs_quantities_list[1],
    "dataset_name":model_list[1].name,
    "file_label":file_label_list[1]
    })
#

# Initialize Settings
settings = scb.FilterSettings(
    initial_covariance=state_covariance,
    process_noise=scb.ProcessNoiseSettings(type='SNC', Q_cont=Q_cont),
    output=scb.OutputSettings(
        metadata={'version': '1.0', 'producer': 'ops_team'}, 
    )
)

srif_it1 = scb.SRIF(propagator=prop_pert_it1,
    settings=settings,
    measurements=measurements_list_new,
    traj_name="orbiter_ref_it1.bsp",
)

solution, n_iters, converged = srif_it1.fit(
    max_iterations=1,
    convergence_threshold=filter_convergence_thr,
    verbose=True,
    traj_name="orbiter_ref.bsp",
)




NameError: name 'pos_0' is not defined

## 1.6 - Visualize results

### Pre- and Post-fit Residuals
Pre-fit residuals are computed before each measurement update using the current reference trajectory, so they reflect how well the propagated orbit predicts the observations. Post-fit residuals are recomputed after the state update and should be smaller and more randomly distributed if the filter is working correctly. Systematic structure in the post-fits (trends, oscillations) indicates unmodeled dynamics or a biased measurement model.

### State Errors
The state error at each epoch is the difference between the filter's absolute state estimate (`solution.state_est`) and the truth trajectory queried from SPICE at the same epoch. Plotting the Euclidean norm of the position and velocity errors alongside the 3σ covariance bound shows whether the filter is *consistent* — a well-tuned filter keeps the errors inside the covariance envelope most of the time.

In [7]:
# ── Pre/post-fit residuals ────────────────────────────────────────────────────
dataset_names_list = list(solution.prefits.keys())
dataset_sr  = dataset_names_list[0]
dataset_dop = dataset_names_list[1]

pre_sr   = np.array(solution.prefits[dataset_sr])    # (N,2): col0=resid, col1=sigma
post_sr  = np.array(solution.postfits[dataset_sr])
pre_dop  = np.array(solution.prefits[dataset_dop])
post_dop = np.array(solution.postfits[dataset_dop])

# Time arrays from the cleaned observation epochs (ET → hours from pass start)
t0_et  = sr_obs_1[0].times.values[0]
t_sr   = (sr_obs_1[0].times.values      - t0_et) / 3600.0
t_dop  = (doppler_obs_1[0].times.values - t0_et) / 3600.0

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle("Pre- and Post-fit Measurement Residuals", fontsize=13)

sr_color  = 'steelblue'
dop_color = 'darkorange'

# Slice time array to residual length inside loop — post-fits may be shorter
# if the filter rejected measurements mid-pass
for ax, res, label in zip(axes[0], [pre_sr, post_sr], ["Pre-fit", "Post-fit"]):
    t_plot = t_sr[:len(res)]
    ax.scatter(t_plot, res[:, 0], s=12, color=sr_color, zorder=3, label='Residual')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(f"{label} — Sequential Ranging  (n={len(res)})")
    ax.set_xlabel("Time from pass start [hr]")
    ax.set_ylabel("Residual [RU]")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for ax, res, label in zip(axes[1], [pre_dop, post_dop], ["Pre-fit", "Post-fit"]):
    t_plot = t_dop[:len(res)]
    ax.scatter(t_plot, res[:, 0], s=12, color=dop_color, zorder=3, label='Residual')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(f"{label} — Doppler  (n={len(res)})")
    ax.set_xlabel("Time from pass start [hr]")
    ax.set_ylabel("Residual [Hz]")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── State errors vs 3σ covariance envelope ────────────────────────────────────
# Extract ET floats from solution.timestamps (stored as ArrayWUnits)
t_sol   = np.array([float(np.asarray(t).ravel()[0]) for t in solution.timestamps])
t_sol_h = (t_sol - t0_et) / 3600.0

# deviation_est == state error because ic_pos_delta = ic_vel_delta = 0
dev = solution.deviation_est                           # (N, 6) km / km/s

pos_err_norm = np.linalg.norm(dev[:, 0:3], axis=1)    # km
vel_err_norm = np.linalg.norm(dev[:, 3:6], axis=1) * 1e3  # m/s

# 3σ from sqrt(trace) of position / velocity sub-block of each covariance
pos_3sig = 3 * np.array([np.sqrt(np.trace(P[0:3, 0:3])) for P in solution.covariance_est])
vel_3sig = 3 * np.array([np.sqrt(np.trace(P[3:6, 3:6])) for P in solution.covariance_est]) * 1e3

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle("State Estimation Errors vs. 3σ Covariance Bound", fontsize=13)

axes[0].semilogy(t_sol_h, pos_err_norm, color='royalblue',  lw=1.5, label=r'$\|\Delta r\|$')
axes[0].semilogy(t_sol_h, pos_3sig,     color='k', lw=1.2, ls='--', label=r'3$\sigma$ bound')
axes[0].set_ylabel("Position error [km]")
axes[0].legend()
axes[0].grid(True, alpha=0.3, which='both')

axes[1].semilogy(t_sol_h, vel_err_norm, color='darkorange', lw=1.5, label=r'$\|\Delta v\|$')
axes[1].semilogy(t_sol_h, vel_3sig,     color='k', lw=1.2, ls='--', label=r'3$\sigma$ bound')
axes[1].set_ylabel("Velocity error [m/s]")
axes[1].set_xlabel("Time from pass start [hr]")
axes[1].legend()
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()


NameError: name 'solution' is not defined